In [ ]:
import pandas as pd
import os
from lamma_ask import run_llama
import re
import glob

# Folder containing your CSVs
folder_path = 'datatables/'

# Get list of all CSV files in the folder
csv_files = glob.glob(os.path.join(folder_path, '*.csv'))

# Read and combine all CSV files
main = pd.concat((pd.read_csv(f) for f in csv_files), ignore_index=True)
main = main.loc[:, ~main.columns.str.contains('^Unnamed')]

#df_combined=df_combined.drop_duplicates()

main.loc[main["video_names"] == "qYbBUXbTn9c", "subtitles"][0]

In [ ]:
final_df = pd.read_csv('../how2QA/questions/how2QA_test_public_release.csv', )
ain = final_df.fillna('')
final_df.columns = ["video_names", "duration", "option1", "option2", "option3", "question", "answer"]
subset_df_1 = final_df[["video_names", "option1", "option2", "option3",	"question",	"answer"]]
subset_df_2=main[["video_names", "option1", "option2", "option3",	"question",	"answer"]]
diff_df = pd.concat([subset_df_1, subset_df_2, subset_df_2]).drop_duplicates(keep=False)
remaining_videos_to_process= list(diff_df["video_names"].unique())
print(len(remaining_videos_to_process))

In [ ]:
import shutil
search_dir = "videos/"
destination_dir = "videos/remaining/"

for root, dirs, files in os.walk(search_dir):
    for file in files:
        if str(file.split('.')[0]) in remaining_videos_to_process:
            src_path = os.path.join(root, file)
            dst_path = os.path.join(destination_dir, file)

            # If duplicate exists, rename to avoid overwrite
            #if os.path.exists(dst_path):
                #base, ext = os.path.splitext(file)
                #counter = 1
                #while os.path.exists(dst_path):
                    #new_name = f"{base}_{counter}{ext}"
                    #dst_path = os.path.join(destination_dir, new_name)
                    #counter += 1

            shutil.move(src_path, dst_path)
            print(f"Moved: {src_path} → {dst_path}")
            
file_count = sum(
    1 for f in os.listdir(destination_dir)
    if os.path.isfile(os.path.join(destination_dir, f))
)

print(f"Total files: {file_count}")

In [ ]:
import ast
i=0
for index, row in main.iterrows():
  #if 'Seq' in q:
    question = main.at[index, 'question']
    answer = main.at[index, "answer"]
    data_str = main.at[index, 'subtitles']
    data = ast.literal_eval(data_str)
    subtitles= [item['text'] for item in data]
    context = main.at[index, 'OIC_context']
    regex_list = [r"\n[A-Za-z0-9_]+ has hair_[A-Za-z0-9_]+",r"\n[A-Za-z0-9_]+ has mouth_[A-Za-z0-9_]+",r"\n[A-Za-z0-9_]+ has [A-Za-z0-9_]+", r"\n[A-Za-z0-9_]+ has ear_[A-Za-z0-9_]+", r"\n[A-Za-z0-9_]+ has head_[A-Za-z0-9_]+", r"\n[A-Za-z0-9_]+ has nose_[A-Za-z0-9_]+", r"\nmouth_[A-Za-z0-9_]+ of [A-Za-z0-9_]+", r"\nhair_[A-Za-z0-9_]+ of [A-Za-z0-9_]+", r"\nhead_[A-Za-z0-9_]+ of [A-Za-z0-9_]+", r"\neye_[A-Za-z0-9_]+ of [A-Za-z0-9_]+", r"\nnose_[A-Za-z0-9_]+ of [A-Za-z0-9_]+",  r"\n[A-Za-z0-9_]+ has arm_[A-Za-z0-9_]+", r"\n[A-Za-z0-9_]+ has face_[A-Za-z0-9_]+", "[0-9]+ seconds:\nAfter"]

    #context_list = set(context.split('\n'))
    #for regex in regex_list:
      #for c in context_list:
        #replaced_text = re.sub(regex,'', context, re.MULTILINE)
        #context = replaced_text

    
    prompt = context
    
    if len(prompt)>len("The scene opens with ")+50:
    
      formatted_question = main.at[index, 'OIC_question']
      
      print('-'*100)
      print(i)
      print(context)
      print(subtitles)
      
      response = run_llama("Video transcript: {} \n Subtitles: {}".format(prompt,subtitles), formatted_question)
      #response = "4"
      main.at[index, 'OIC_answer_llama'] = str(response)
      main.at[index, 'OIC_context'] = prompt
      OIC_answer = response
      print('OIC question: {}'.format(formatted_question))
      print('OIC answer: {}'.format(OIC_answer))
      if len(OIC_answer)>1:
        main.at[index, 'Match_llama'] = OIC_answer
      else:
        if int(OIC_answer) == 4: #every 4th options are correct
            main.at[index, 'Match_llama'] = 'Correct'
            print('correct')
        else:
            print('wrong')
            main.at[index, 'Match_llama'] = 'Wrong'
      
      main.to_csv('for_eval/OIC_llama_with_st.csv')
      i=i+1
    

import re

list_answers = []
# Read the video description from a text file
with open('/.../.../Developer/IMP-OIC/LifeQA/datatables/output.txt', 'r') as file:
    
    for line in file:
        match = re.search(r'OIC answer: (.+?)(?:\n|$)', line)
        
        if match:
            answer = match.group(1)
            list_answers.append(answer)
            print(answer)

count=505  
for l in list_answers:       
    
    
    main.loc[main.index == count, 'OIC_answer'] = l
    
    count+=1

main.to_csv('LifeQA/for_eval/OIC_gpt4_wo_st_updated.csv')

main.loc[main.index == 1796, 'OIC_answer']

new_pd = pd.read_csv('/.../.../Developer/IMP-OIC/LifeQA/for_eval/OIC_gpt4_wo_st_updated.csv')
new_pd.loc[508,:]['OIC_answer']

print(list_answers)